In [5]:
import sys
import maboss
import pandas as pd
import numpy as np
sys.path.append("/Users/emilieyu/endotehelial-masboss")

In [6]:
from src.config import load_sim_config, load_sweep_config
from src.paths import *
from boolean_models.scripts import run_perturbations, run_sweeps

sim_cfg = load_sim_config()
sweep_cfg = load_sweep_config()

MODELS_BND = DATA_DIR / sim_cfg['model']['bnd']
MODELS_CFG = DATA_DIR / sim_cfg['model']['cfg_v3']

base_model = maboss.load(str(MODELS_BND), str(MODELS_CFG))
base_model.param['max_time'] = 10.0
base_model.param['sample_count'] = 5000


In [ ]:
from abm.flow_field import FlowField 

## Sigmoid Parameter Activation

In [49]:
def sigmoid(x, threshold, steepness=5.0):
    return 1 / (1.0 + np.exp(-steepness * (x - threshold)))

field = FlowField()
positions = np.array([[0, 0],
                      [1, 0.8],
                      [1, -0.8], 
                      [2, 1.5], 
                      [2, -1.5], 
                      [3, 1],
                      [3, -1], 
                      [4, 0]])
print(f"positions: {positions}")
direction=np.array([1.0, 0.0])
shear_rate=1.0

positions: [[ 0.   0. ]
 [ 1.   0.8]
 [ 1.  -0.8]
 [ 2.   1.5]
 [ 2.  -1.5]
 [ 3.   1. ]
 [ 3.  -1. ]
 [ 4.   0. ]]


In [50]:
centre = positions.mean(axis=0)
print(f"centre: {centre}")
relative = positions - centre
print(f"relative: {relative}")
norms = np.linalg.norm(relative, axis=1, keepdims=True) 
print(f"norms: {norms}")
outward_normals = relative / (norms + 1e-10) 
print(f"outward_normals {outward_normals}")
alignment = outward_normals @ direction
print(f"alignment {alignment}")

centre: [2. 0.]
relative: [[-2.   0. ]
 [-1.   0.8]
 [-1.  -0.8]
 [ 0.   1.5]
 [ 0.  -1.5]
 [ 1.   1. ]
 [ 1.  -1. ]
 [ 2.   0. ]]
norms: [[2.        ]
 [1.28062485]
 [1.28062485]
 [1.5       ]
 [1.5       ]
 [1.41421356]
 [1.41421356]
 [2.        ]]
outward_normals [[-1.          0.        ]
 [-0.78086881  0.62469505]
 [-0.78086881 -0.62469505]
 [ 0.          1.        ]
 [ 0.         -1.        ]
 [ 0.70710678  0.70710678]
 [ 0.70710678 -0.70710678]
 [ 1.          0.        ]]
alignment [-1.         -0.78086881 -0.78086881  0.          0.          0.70710678
  0.70710678  1.        ]


In [51]:
face_types = np.where(alignment < -0.6, 'upstream', np.where(alignment > 0.6, 'downstream', 'lateral'))
face_types

array(['upstream', 'upstream', 'upstream', 'lateral', 'lateral',
       'downstream', 'downstream', 'downstream'], dtype='<U10')

In [56]:
tangent = np.array([-outward_normals[3][1], outward_normals[3][0]])
tangent
alignment_to_flow = np.dot(direction, tangent)
magnitude = shear_rate * alignment_to_flow
print(f"magnitude: {magnitude}")
force_vector = magnitude * direction
print(f"force_vector {force_vector}")

magnitude: -0.9999999999333333
force_vector [-1. -0.]


## Testing DataStructures

In [53]:
positions = np.ndarray(2)
positions

array([-0., -1.])

## Run Perturbation and Get Result

In [3]:
perb_df, ss_df = run_perturbations(base_model, sim_cfg)

DEBUG: Running perturbation: WT
DEBUG: Running perturbation: DSP_KO
DEBUG: Running perturbation: TJP1_KO
DEBUG: Running perturbation: JCAD_KO
DEBUG: Running perturbation: DSP_JCAD_DKO
DEBUG: Running perturbation: TJP1_JCAD_DKO
DEBUG: All simulations completed successfully


In [4]:
ss_df

,t,DSP,TJP1,JCAD,RhoA,RhoC,perturbation,delta,phenotype
0,9.9,0.000000,0.498400,0.000000,0.395196,0.558345,DSP_JCAD_DKO,0.163149,Normal
1,9.9,0.000000,0.492999,0.491800,0.353242,0.627831,DSP_KO,0.274589,Hyper
2,9.9,0.506201,0.485401,0.000000,0.506329,0.501611,JCAD_KO,-0.004718,Normal
3,9.9,0.503400,0.000000,0.000000,0.566635,0.394151,TJP1_JCAD_DKO,-0.172484,Normal
4,9.9,0.496400,0.000000,0.496600,0.634992,0.353916,TJP1_KO,-0.281076,Failed
5,9.9,0.510399,0.500000,0.517999,0.542614,0.552133,WT,0.009519,Normal


In [ ]:
def sigmoid(x, threshold, steepness=5.0):
    return 1 / (1.0 + np.exp(-steepness * (x - threshold)))